In [ ]:
%run bootstrap.ipynb

# THE VACUUM WORLD   

In this notebook, we will be discussing **the structure of agents** through an example of the **vacuum agent**. The job of AI is to design an **agent program** that implements the agent function: the mapping from percepts to actions. We assume this program will run on some sort of computing device with physical sensors and actuators: we call this the **architecture**:

<h3 align="center">agent = architecture + program</h3>

Before moving on, please review [<b>agents.ipynb</b>](https://github.com/aimacode/aima-python/blob/master/agents.ipynb)

In [ ]:
import random

from aima.agents import *
from aima.mdp import GridMDP, best_policy, value_iteration
from aima.notebook_utils import psource

## MPD VACUUM AGENT

Environment Set-up:

In [ ]:
ACTION_STR = {
    (0, 1): "Up",
    (0, -1): "Down",
    (1, 0): "Right",
    (-1, 0): "Left",
    None: "NoOpt",
}

START = (3, 3)
ENVIRONMENT = [
    [0, 0, 0, 0],
    [0, None, 0, 0],
    [0, 0, 0, 0],
    [0, -1, 0, 10],
]

def find_location(marker):
    locations = [
        (x, len(ENVIRONMENT) - row_index - 1)
        for row_index, row in enumerate(ENVIRONMENT)
        for x, cell in enumerate(row)
        if cell == marker
    ]
    if len(locations) != 1:
        raise Exception(
            f"Expected exactly one cell marked {marker}, found {len(locations)}"
        )
    return locations[0]

MPD Agent:

In [ ]:
RECHARGE = find_location(10)
TRAP = find_location(-1)


def build_mdp(step_cost):
    grid = [[step_cost if cell == 0 else cell for cell in row] for row in ENVIRONMENT]
    return GridMDP(grid, terminals=[RECHARGE, TRAP], gamma=0.95)


def run_case(step_cost, max_steps=100):
    print(f"Living Cost: {step_cost}")
    mdp = build_mdp(step_cost)
    U = value_iteration(mdp, epsilon=1e-4)
    pi = best_policy(mdp, U)

    print("Policy grid:")
    for y in range(3, -1, -1):
        row = []
        for x in range(4):
            s = (x, y)
            if s not in mdp.states:
                row.append("  X   ")
            else:
                a = ACTION_STR.get(pi.get(s), "-")
                row.append(f"{U[s]:5.2f}{a[0]}")
        print(" ".join(row))

    loc = START
    print("Path followed:")
    print(f"  step 00: at {loc}")
    steps = 0
    while loc not in (RECHARGE, TRAP) and steps < max_steps:
        mdp_action = pi.get(loc)
        if mdp_action is None:
            break
        action = ACTION_STR[mdp_action]
        outcomes = mdp.T(loc, mdp_action)
        loc = random.choices(
            [next_state for _, next_state in outcomes],
            weights=[probability for probability, _ in outcomes],
            k=1,
        )[0]
        steps += 1
        print(f"  step {steps:02d}: {action:5s} -> {loc}")
    if loc == RECHARGE:
        print(f"RESULT: reached recharge in {steps} steps ✓\n")
    elif loc == TRAP:
        print(f"RESULT: got stuck in trap after {steps} steps ✗\n")
    else:
        print(f"RESULT: ended at {loc} after {steps} steps (max reached)\n")


## Tests

In [ ]:
living_costs = [-0.02, -0.1, -0.5]
for c in living_costs:
    run_case(c)